In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql import functions as F
import time
from pyspark.sql.functions import col, when, to_date, year, month, dayofmonth, trim, lower, round as spark_round,avg,count
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import time
import os
import shutil
from pyspark.sql import SparkSession

# ============================================
# SETUP
# ============================================
try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("CSV vs Parquet") \
    .master("local[4]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

# Verify configuration
print(f"Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"Shuffle Partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"Master URL: {spark.conf.get('spark.master')}")


Driver Memory: 8g
Shuffle Partitions: 8
Master URL: local[4]


In [3]:
csv_path = '/opt/spark/data/'
df_csv = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(csv_path)
print(f"Number of partitions: {df_csv.rdd.getNumPartitions()}")
print(f"Number of rows: {df_csv.count()}")
df_csv.printSchema()

Number of partitions: 10
Number of rows: 2830743
root
 |--  Destination Port: integer (nullable = true)
 |--  Flow Duration: integer (nullable = true)
 |--  Total Fwd Packets: integer (nullable = true)
 |--  Total Backward Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: integer (nullable = true)
 |--  Total Length of Bwd Packets: integer (nullable = true)
 |--  Fwd Packet Length Max: integer (nullable = true)
 |--  Fwd Packet Length Min: integer (nullable = true)
 |--  Fwd Packet Length Mean: double (nullable = true)
 |--  Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |--  Bwd Packet Length Min: integer (nullable = true)
 |--  Bwd Packet Length Mean: double (nullable = true)
 |--  Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |--  Flow Packets/s: double (nullable = true)
 |--  Flow IAT Mean: double (nullable = true)
 |--  Flow IAT Std: double (nullable = true)
 |

In [4]:
df_csv.limit(5).toPandas().head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [5]:
df_csv.select(col(df_csv.columns[-1])).distinct().show()

+--------------------+
|               Label|
+--------------------+
|              BENIGN|
|            DoS Hulk|
|       DoS slowloris|
|    DoS Slowhttptest|
|         FTP-Patator|
|         SSH-Patator|
|       DoS GoldenEye|
|          Heartbleed|
|        Infiltration|
|                DDoS|
|            PortScan|
|Web Attack � Brut...|
|    Web Attack � XSS|
|Web Attack � Sql ...|
|                 Bot|
+--------------------+



In [6]:
df_csv.groupBy(df_csv.columns[-1]).count().orderBy('count',ascending=False).show()

+--------------------+-------+
|               Label|  count|
+--------------------+-------+
|              BENIGN|2273097|
|            DoS Hulk| 231073|
|            PortScan| 158930|
|                DDoS| 128027|
|       DoS GoldenEye|  10293|
|         FTP-Patator|   7938|
|         SSH-Patator|   5897|
|       DoS slowloris|   5796|
|    DoS Slowhttptest|   5499|
|                 Bot|   1966|
|Web Attack � Brut...|   1507|
|    Web Attack � XSS|    652|
|        Infiltration|     36|
|Web Attack � Sql ...|     21|
|          Heartbleed|     11|
+--------------------+-------+



In [7]:
new_col = (c.strip() for c in df_csv.columns)
df_clean = df_csv.toDF(*new_col)
df_clean.printSchema()

root
 |-- Destination Port: integer (nullable = true)
 |-- Flow Duration: integer (nullable = true)
 |-- Total Fwd Packets: integer (nullable = true)
 |-- Total Backward Packets: integer (nullable = true)
 |-- Total Length of Fwd Packets: integer (nullable = true)
 |-- Total Length of Bwd Packets: integer (nullable = true)
 |-- Fwd Packet Length Max: integer (nullable = true)
 |-- Fwd Packet Length Min: integer (nullable = true)
 |-- Fwd Packet Length Mean: double (nullable = true)
 |-- Fwd Packet Length Std: double (nullable = true)
 |-- Bwd Packet Length Max: integer (nullable = true)
 |-- Bwd Packet Length Min: integer (nullable = true)
 |-- Bwd Packet Length Mean: double (nullable = true)
 |-- Bwd Packet Length Std: double (nullable = true)
 |-- Flow Bytes/s: double (nullable = true)
 |-- Flow Packets/s: double (nullable = true)
 |-- Flow IAT Mean: double (nullable = true)
 |-- Flow IAT Std: double (nullable = true)
 |-- Flow IAT Max: integer (nullable = true)
 |-- Flow IAT Min: in

In [8]:
df_clean.limit(5).toPandas().head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [9]:
duplicates = df_clean.count() - df_clean.dropDuplicates().count()
print(f"the nomber of duplicates in the dataset is :{duplicates}")

the nomber of duplicates in the dataset is :308381


In [10]:
df_clean = df_clean.dropDuplicates()
duplicates = df_clean.count() - df_clean.dropDuplicates().count()
print(f"the nomber of duplicates in the dataset is :{duplicates}")

the nomber of duplicates in the dataset is :0


In [11]:
from pyspark.sql.functions import col, sum as spark_sum

# Calculate null counts
null_counts = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) 
    for c in df_clean.columns
]).collect()[0]

# Print column by column
print("Null counts per column:")
print("-" * 40)
for col_name in df_clean.columns:
    count = null_counts[col_name]
    print(f"{col_name:30s}: {count}")

Null counts per column:
----------------------------------------
Destination Port              : 0
Flow Duration                 : 0
Total Fwd Packets             : 0
Total Backward Packets        : 0
Total Length of Fwd Packets   : 0
Total Length of Bwd Packets   : 0
Fwd Packet Length Max         : 0
Fwd Packet Length Min         : 0
Fwd Packet Length Mean        : 0
Fwd Packet Length Std         : 0
Bwd Packet Length Max         : 0
Bwd Packet Length Min         : 0
Bwd Packet Length Mean        : 0
Bwd Packet Length Std         : 0
Flow Bytes/s                  : 0
Flow Packets/s                : 0
Flow IAT Mean                 : 0
Flow IAT Std                  : 0
Flow IAT Max                  : 0
Flow IAT Min                  : 0
Fwd IAT Total                 : 0
Fwd IAT Mean                  : 0
Fwd IAT Std                   : 0
Fwd IAT Max                   : 0
Fwd IAT Min                   : 0
Bwd IAT Total                 : 0
Bwd IAT Mean                  : 0
Bwd IAT Std      

In [12]:
from pyspark.sql.functions import col, when, isnan

# Replace inf and -inf with null for all numeric columns
for c in df_clean.columns:
    df_clean = df_clean.withColumn(c, 
        when(col(c).isin([float('inf'), float('-inf')]), None)
        .otherwise(col(c))
    )

In [13]:
from pyspark.sql.functions import regexp_replace

df_clean = df_clean.withColumn(
    "Label",
    regexp_replace(col("Label"), "[^\\x00-\\x7F]", "")
)

In [14]:
from pyspark.sql.functions import regexp_replace, col, trim, when

label_col = trim(regexp_replace(col("Label"), "\\s+", " "))

In [15]:
df_clean.select("Label").distinct().show(truncate=False)

+-------------------------+
|Label                    |
+-------------------------+
|BENIGN                   |
|DoS Hulk                 |
|DoS slowloris            |
|DoS Slowhttptest         |
|FTP-Patator              |
|SSH-Patator              |
|DoS GoldenEye            |
|Heartbleed               |
|Infiltration             |
|DDoS                     |
|PortScan                 |
|Web Attack  Sql Injection|
|Web Attack  XSS          |
|Bot                      |
|Web Attack  Brute Force  |
+-------------------------+



In [17]:
drop_cols = [
    'Destination Port',         # duplicate column
    'Fwd Header Length.1',      # useless flags (almost always 0)
    'Fwd PSH Flags',
    'Bwd PSH Flags',
    'Fwd URG Flags',
    'Bwd URG Flags',
    'URG Flag Count',
    'CWE Flag Count',
    'ECE Flag Count',           # broken CICFlowMeter bulk features
    'Fwd Avg Bytes/Bulk',
    'Fwd Avg Packets/Bulk',
    'Fwd Avg Bulk Rate',
    'Bwd Avg Bytes/Bulk',
    'Bwd Avg Packets/Bulk',
    'Bwd Avg Bulk Rate',        # redundant (duplicate of totals)
    'Subflow Fwd Packets',
    'Subflow Fwd Bytes',
    'Subflow Bwd Packets',
    'Subflow Bwd Bytes',        # weak / noisy features
    'Init_Win_bytes_forward',
    'Init_Win_bytes_backward',
]

# Only drop columns that actually exist in the dataframe
cols_to_drop = [c for c in drop_cols if c in df_clean.columns]
df_clean = df_clean.drop(*cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns, {len(df_clean.columns)} remaining")

Dropped 20 columns, 59 remaining


In [18]:
df_clean = df_clean.withColumn("Label",
    when(label_col == "BENIGN",                    "Normal")
    .when(label_col == "DoS Hulk",                 "DoS/DDoS")
    .when(label_col == "DoS GoldenEye",            "DoS/DDoS")
    .when(label_col == "DoS slowloris",            "DoS/DDoS")
    .when(label_col == "DoS Slowhttptest",         "DoS/DDoS")
    .when(label_col == "DDoS",                     "DoS/DDoS")
    .when(label_col == "FTP-Patator",              "BruteForce")
    .when(label_col == "SSH-Patator",              "BruteForce")
    .when(label_col == "Web Attack Brute Force",   "WebAttack")
    .when(label_col == "Web Attack XSS",           "WebAttack")
    .when(label_col == "Web Attack Sql Injection", "WebAttack")
    .when(label_col == "Bot",                      "Botnet")
    .when(label_col == "PortScan",                 "Recon")
    .when(label_col == "Infiltration",             "Infiltration")
    .when(label_col == "Heartbleed",               "Heartbleed")
    .otherwise("Unknown")   # ← important
)

In [19]:
df_clean.select(col(df_clean.columns[-1])).distinct().show()

+------------+
|       Label|
+------------+
|    DoS/DDoS|
|       Recon|
|Infiltration|
|  BruteForce|
|      Normal|
|      Botnet|
|   WebAttack|
|  Heartbleed|
+------------+



In [20]:
df_clean.write.parquet("data/clean/", mode="overwrite")

In [21]:
spark.stop()